# Generate Figure in style of Figure 2 of Kaiser et al. 2020 ( log(K/Ca) vs. log(Na/Ca) ) but for any element combo

This is the same message at the beginning of all jupyter notebooks in this directory. 

If you don't have the below packages, you obviously need to install them for this to work. If it doesn't work still it's extremely likely you have an outdated version of one of the packages. Alternatively, some of the histogram functions actually rely on not being the most recent version because they changed from "normed" to something else from my recollection. Or perhaps it was the other way. I am aware this was poor decision-making, but it works (if you use the right version). ¯\\_(ツ)_/¯

Also pretty much all of these commands are copied and pasted from another Jupyter notebook I made but contained tons of tries at doing this stuff (and unrelated efforts) so that's why a lot of the variables seem unnecessary to use.

Ok, we basically need to follow the convention of Swan et al. 2019 and create arrows whose length is equal to some multiple of the "e-folding" time. 

## 2021-10-08 The flexibility for modelers, atm_type, and overshoot have been added. Also, the decreasing phase arrow error has been corrected.

In [1]:
from __future__ import print_function

import matplotlib

matplotlib.use('pdf')
savefig=True
    
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from astropy.io import fits
from glob import glob
from astropy.time import Time
from astropy import coordinates as coords
from astropy import units as u
from astropy import constants as const
from astropy import convolution as conv
from astropy.table import Table, Column
import scipy.interpolate as scinterp
import time
import periodictable as pt

start = time.time()
print(start)
time_string=str(start).split('.')[0]

#from mendeleev import O, Ca, Li, Na, Si, Fe, Mg, He
start = time.time()

#import wdatmos
import spec_plot_tools as spt
import cal_params as cp
import plot_spec as ps
import abundance_corrections as acorr
import interp_tau as itau
import fix_strings as fs
import nuclear_cross_sections as nc


#print(os.getcwd())

1675971075.521821
all_avg
(116, 4, 27)
(4, 27, 116)
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DA_diff_ov00_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DA_diff_ov10_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DB_diff_ov00_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DB_diff_ov10_diffusion_timescales.csv


In [2]:
plt.show()

In [3]:
figure_output_dir='/Users/BenKaiser/Desktop/'
#figure_output_dir='/Users/BenKaiser/Desktop/z_plots_for_Hollands/'
#figure_output_dir='/Users/BenKaiser/Desktop/GaiaJ1644m0449_paper/ApJ_reformat/figures'

In [4]:
target_dir= '/Users/BenKaiser/Desktop/radial_velocity_calculations/'
os.chdir(target_dir)

In [5]:
#wd_abund_file='all_wd_abundances.csv'
#wd_abund_file='20210131_all_wd_abundances.csv'
#wd_abund_file='20210713_all_wd_abundances_bedard_cooling_no_WDs_plot.csv'
#wd_abund_file='20210818_all_wd_abundances_bedard_cooling_newer_Blouin.csv'
#wd_abund_file='20211112_all_wd_abundances_beryllium_objects_partially_added.csv'
#wd_abund_file='20220222_all_wd_abundances_newMC_ages.csv'
#wd_abund_file='20220303_all_wd_abundances_newMC_ages_allCa_abunds.csv'
#wd_abund_file='20220926_all_wd_abundances_7030MCages.csv'
wd_abund_file='20221109_all_wd_abundances_7030MCages_DR3kinematics_missingabundsJ1636added.csv'
#wd_abund_file='20220718_fake_wd_abundances_get_arrow_length.csv' #file created to determine arrow lengths from J1644 for all elements relative to Ca

lodders_abund_file='Lodders2009_solarsystem_abundances.csv'
#solar_system_object_file='solar_system_body_abundances.csv'
#solar_system_object_file='solar_system_body_abundances_mgfe_fixed.csv'
#solar_system_object_file='solar_system_body_abundances_sea_added.csv'
#solar_system_object_file='20210713_solar_system_body_abundances_show_name_added.csv'
#solar_system_object_file='20210713_solar_system_body_abundances_all_names.csv'
#solar_system_object_file='20220304_solar_system_body_abundances_all_names.csv'
solar_system_object_file='20220304_solar_system_body_abundances_select_names.csv'




wd_num_abund_file='20220204_beryllium_WD_linear_number_abundances.csv'


In [6]:
print(os.getcwd())

print(wd_abund_file)
wd_abund_table=Table.read(wd_abund_file)
wd_abund_table=spt.clean_color_string(wd_abund_table,color_header='plot_color')
lodders_table=Table.read(lodders_abund_file)
bodies_table=Table.read(solar_system_object_file)
lodders_table.add_index('element')
wd_abund_table.add_index('name')
bodies_table.add_index('name')

wd_num_abund_table=Table.read(wd_num_abund_file)
wd_num_abund_table.add_index('name')

limit_length=0.3 #length of limit error bars on plots
limit_indicator=99. #value above which if the absolute value of the error on a measurement is above it indicates it should be a limit


/Users/BenKaiser/Desktop/radial_velocity_calculations
20221109_all_wd_abundances_7030MCages_DR3kinematics_missingabundsJ1636added.csv


In [7]:
use_indices=np.where(bodies_table['show']==1)
use_bodies_table=bodies_table[use_indices]

In [8]:
spall_n_sigma=1.

t_step=5
#t_step=1

wd_marker='*'
#met_marker='D'
#ssp_marker='met_marker'
met_marker='s'
ssp_marker='D'
met_color='#1ca1f2'
met_size=3
ci_size=6
wd_size=10
dp_alpha=0.5
ci_leg_size=9
arr_naca=[-0.4,-0.1]
alpha_range=[0.5,0.2]
arrow_segs=100
arrow_width=0.03
#arrow_width=0.07

arrow_line=4
figure_text_size=6
default_offset=[0.05,0.00]
annot_line_weight=0.03

show_all_ssobj_names=True
default_cross_source='read' #source of nuclear cross-sections for calculation of implied spallation levels


In [9]:
def add_arrow(line, position=None, direction='right', size=20, color=None, slope=None):
    """
    add an arrow to a line.

    line:       Line2D object
    position:   x-position of the arrow. If None, mean of xdata is taken
    direction:  'left' or 'right'
    size:       size of the arrow in fontsize points
    color:      if None, line color is taken.
    """
    if color is None:
        color = line.get_color()

    xdata = line.get_xdata()
    ydata = line.get_ydata()

    if position is None:
        position = xdata.mean()
    # find closest index
    start_ind = np.argmin(np.absolute(xdata - position))
    if direction == 'right':
        end_ind = start_ind + 1
    else:
        end_ind = start_ind - 1
    #xend=0.5*(xdata[end_ind]-xdata[start_ind])+xdata[start_ind]
    #yend=0.5*(ydata[end_ind]-ydata[end_ind])+ydata[start_ind]
    if slope is None:
        print('no slope...')
        xend=xdata[end_ind]
        yend=ydata[end_ind]
        xstart=xdata[start_ind]
        ystart=ydata[start_ind]
    else:
        #xend=0.5*(xdata[end_ind]-xdata[start_ind])+xdata[start_ind]
        xend=xdata[end_ind]
        #yend=slope*(xend-xdata[start_ind])+ydata[start_ind] #slope-intercept form that should work
        xstart=0.5*(xdata[start_ind]-xdata[end_ind])+xdata[end_ind]
        ystart=slope*(xstart-xdata[end_ind])+ydata[end_ind] #slope-intercept form that should work
    line.axes.annotate('',
        #xytext=(xdata[start_ind], ydata[start_ind]),
        xytext=(xstart,ystart),
        xy=(xdata[end_ind], ydata[end_ind]),
        #xy=(xend, yend),
        arrowprops=dict(arrowstyle="<|-", color=color),
        size=size
    )




I need a function to calculate the combined error on two elements put together using the equation (A2) from Klein et al. 2021. This is only possible when the uncertainty due to spread and uncertainty due to T_eff are presented separately

In [10]:
def get_BeCa(wd_name='GALEXJ2339-0424',n_sigma=1.):
    BeCa,BeCa_lo, BeCa_hi= acorr.get_el1el2_full_err(wd_num_abund_table.loc[wd_name],'be','ca',n_sigma=n_sigma)
    return BeCa, BeCa_lo, BeCa_hi

print(get_BeCa())
print(get_BeCa(n_sigma=3.))

be/ca 0.004361702127659575 +/- 0.0020716256896796434
log10(be/ca): -2.3603439968799633 ,upper bound: -2.19156431807979 ,lower bound: -2.6401500215732696
(-2.3603439968799633, -2.6401500215732696, -2.19156431807979)
be/ca 0.004361702127659575 +/- 0.0020716256896796434
log10(be/ca): -2.3603439968799633 ,upper bound: -1.9756547742878527 ,lower bound: nan
(-2.3603439968799633, nan, -1.9756547742878527)


/Users/BenKaiser/Desktop/radial_velocity_calculations/abundance_corrections.py:61: RuntimeWarning: invalid value encountered in log10
  log_lo_bound=np.log10(lower_bound)


Now I want to be able to use that output as an input into the relative reaction rate stuff to yield an estimate of the Li/Ca you should be able to expect assuming all(-ish) of the beryllium is from spallation of material to be accreted.

In [11]:


lodders_CI_LiCa=lodders_table.loc['Li']["A_el"]-lodders_table.loc['Ca']["A_el"]
print(lodders_CI_LiCa)

-3.03


In [12]:
def get_spalled_LiCa(n_sigma=1.,cross_source=default_cross_source, cross_method='max',spall_product='Li',projectile='p',spall_target='O16'):
    """
    see pages 30-32 of General Clemens XI for algebra for calculation
    """
    BeCa, BeCa_lo, BeCa_hi=get_BeCa(n_sigma=n_sigma)
    BeCa_array=np.array([BeCa,BeCa_lo,BeCa_hi])
    
    
    if cross_source=='crude':
        if spall_product=='Li':
            Li7_cross_section=5e-26 #Li-7 spallation cross-section estimated from Prantzos 2012 figure 10 in units of cm^2
            Li6_cross_section=2e-26 #Li-6 spallation cross-section estimated from Prantzos 2012 figure 10 in units of cm^2
            Be9_cross_section=1e-26 #Be-9 spallation cross-section estimated from Prantzos 2012 figure 10 in units of cm^2
            LiCa_array=np.log10((Li7_cross_section+Li6_cross_section)/Be9_cross_section*10.**BeCa_array+10.**lodders_CI_LiCa)
        else:
            print('cross_source=="crude" only works for lithium (Li) as the product')
    elif cross_source=='read':
        energy_range=np.linspace(1,1000,1000)
        Be9_cross_section_array=nc.get_cross_section(energy_range, projectile=projectile, target=spall_target,product='A9')
        if cross_method=='max':
            max_energy_index=np.nanargmax(Be9_cross_section_array)
            max_cross_energy=energy_range[max_energy_index]
            print('Max Cross Section energy:',max_cross_energy, 'MeV')
            Be9_cross_section=Be9_cross_section_array[max_energy_index]
            if spall_product=='Li':
                Li6_cross_section=nc.get_cross_section(max_cross_energy, projectile=projectile, target=spall_target,product='A6')
                Li7_cross_section=nc.get_cross_section(max_cross_energy, projectile=projectile, target=spall_target,product='A7')
                print('Be-9 Cross Section:',Be9_cross_section)
                print('Li-6 Cross Section:', Li6_cross_section)
                print('Li-7 Cross Section:',Li7_cross_section)
                LiCa_array=np.log10((Li7_cross_section+Li6_cross_section)/Be9_cross_section*10.**BeCa_array+10.**lodders_CI_LiCa)
            elif spall_product=='B':
                print('spall_product==B not implemented yet')
            else:
                print('spall_product not recognized:', spall_product)
        else:
            print("cross_method not recognized:", cross_method)

    print('LiCa_array',LiCa_array)
    return LiCa_array

I think I want a more robust errorbar plotting function that will just determine if the input errors actually should be limits and make sure they are pointing the correct direction. So I think I'm going to restructure the input file to use el1/el2_err = -100 to indicate an upper limit and +100 to indicate a lower limit. Then I'll have 

In [13]:
def plot_wd_errorbar(el3el2, el1el2,  el3el2_err, el1el2_err,row, selected_marker=wd_marker, markersize=wd_size, label='', color='b',show_name=True):
    if label=='':
        #label=name #commented this on 2021-11-12 to try to keep star points out of legend
        pass 
    else:
        pass
    name=row['name']
    if show_name:
        display_name=row['display_name']
    else:
        display_name=''
    uplims=False
    lolims=False
    xlolims=False
    xuplims=False
    if np.abs(el3el2_err)> limit_indicator:
        if el3el2_err > 0:
            xlolims=True
        elif el3el2_err < 0:
            xuplims=True
        else:
            print("This shouldn't print el3el2_err")
        el3el2_err=limit_length
    if np.abs(el1el2_err)> limit_indicator:
        if el1el2_err > 0:
            lolims=True
        elif el1el2_err < 0:
            uplims=True
        else:
            print("This shouldn't print el3el2_err")
        el1el2_err= limit_length
    else:
        #no limit indicators are present for the 2 relative abundances input
        pass
    plt.errorbar(el3el2, el1el2, yerr= el1el2_err, xerr= el3el2_err, uplims=uplims, lolims=lolims, xuplims=xuplims, xlolims=xlolims, color=color,marker=selected_marker,  markersize=markersize,linestyle='None')
    #plt.errorbar(el3el2,el1el2, label=display_name, marker=selected_marker, markersize=markersize, color=color,linestyle='None')
    plt.errorbar(np.nan,np.nan, label=display_name, marker=selected_marker, markersize=markersize, color=color,linestyle='None')
    return

In [14]:

def plot_wd_el1el2el3(row, logg='default', teff='default', t_step=10, naca_min=-4.0, t_max=100, elements=["K","Ca","Na"], t_step_units='Myr',plot_type='line',SSP=True):
    """
    
    
    t_step=10, time in Myr of time-step for declining phase
    
    
    """
    name=row['name']
    print('starting plotting effort for',name)
    string1= elements[0].lower()+'/'+elements[1].lower()
    string2=elements[2].lower()+'/'+elements[1].lower()
    #times= np.arange(0, t_max+t_step, t_step)
    markersize=wd_size
    #target_row=wd_abund_table.loc[name]
    target_row=row
    
    if target_row['show']==0:
        return
    else:
        pass
    target_el1el2=target_row[string1]
    target_el3el2=target_row[string2]
    el1el2_err=target_row[string1+'_err']
    el3el2_err= target_row[string2+'_err']
    label=target_row['name']
    label=fs.fix_display_string(label)
    if logg=='default':
        logg=target_row['logg']
    else:
        pass
    if teff=='default':
        teff= target_row['teff']
    else:
        pass
    #if target_row[string1+'_err']>0.:
    #    plt.errorbar(target_row[string2],target_row[string1],xerr=target_row[string2+'_err'],yerr=target_row[string1+'_err'], marker=wd_marker, markersize=wd_size, color=color_dict[target_row['name']],linestyle='None')
    #    plt.errorbar(target_row[string2],target_row[string1], label=fs.fix_display_string(target_row['name']), marker=wd_marker, markersize=wd_size, color=color_dict[target_row['name']],linestyle='None')
    #elif target_row[string2+'_err'] < 0.001:
    #    plt.errorbar(target_row[string2],target_row[string1],xerr=0.3,yerr=0.3, uplims=True,xuplims=True, marker=wd_marker, color=color_dict[target_row['name']], markersize=wd_size,linestyle='None')
    #    plt.errorbar(target_row[string2],target_row[string1], label=fs.fix_display_string(target_row['name']), marker=wd_marker, color=color_dict[target_row['name']], markersize=wd_size,linestyle='None')
    #else:
    #    plt.errorbar(target_row[string2],target_row[string1],xerr=target_row[string2+'_err'],yerr=0.3, uplims=True, marker=wd_marker, color=color_dict[target_row['name']], markersize=wd_size,linestyle='None')
    #    plt.errorbar(target_row[string2],target_row[string1], label=fs.fix_display_string(target_row['name']), marker=wd_marker, color=color_dict[target_row['name']], markersize=wd_size,linestyle='None')
    
    #plot_wd_errorbar(target_el3el2, target_el1el2,  el3el2_err, el1el2_err,name, selected_marker=wd_marker, markersize=wd_size, label=label+' Photo.', color=target_row['plot_color'])
    plot_wd_errorbar(target_el3el2, target_el1el2,  el3el2_err, el1el2_err,row, selected_marker=wd_marker, markersize=wd_size, color=target_row['plot_color'],show_name=False)
    
    if SSP:
        
        plot_marker=ssp_marker
        markersize=ci_size
        target_el1el2, target_el3el2, el1el2_err, el3el2_err=acorr.easy_dist_ssp(target_row,elements, plot_all=False,tau_rand=True)
        #label=label+' SSP'
        print('SSP log('+elements[0]+'/' +elements[1]+') =',target_el1el2,'+/-',el1el2_err)
        print('SSP log('+elements[2]+'/' +elements[1]+') =',target_el3el2,'+/-',el3el2_err)
        label=label+' Steady State'
    else:
        print('Not SSP!')
        plot_marker=wd_marker
        el1el2_err=target_row[string2+'_err']
        el3el2_err=target_row[string1+'_err']
        #label=label+' Photo.'
        label=''
        
    if t_step_units != 'Myr':
        print("t_step_units is not Myr, meaning it's some diffusion timescale multiple")
        print('so t_step= ',t_step,"* tau_",t_step_units)
        tau_time= 10.**itau.extrapolate_tau_x_logg(teff, logg, t_step_units,atm_type=target_row['diff_atm_type'],modeler=acorr.default_modeler, overshoot=acorr.default_overshoot)
        tau_time=tau_time*1e-6 #converted to Myr
        t_step=t_step*tau_time
        t_max=t_max*tau_time
        print('New t_step:',t_step, 'Myr')
        print('New t_max:', t_max, 'Myr')
    else:
        pass
    if target_row['show_dp']==1:
        if plot_type=='line':
            times= np.arange(-1*t_max-t_step, t_max+t_step, t_step)
            #dp_el1el2, dp_el3el2= acorr.el1el2_DP_el3el2_ftimes(teff, target_row['k/ca'], target_row['na/ca'],times, 'K', "Ca", "Na", logg=logg)
            dp_el1el2, dp_el3el2= acorr.el1el2_DP_el3el2_ftimes(teff, target_row[string1], target_row[string2],times, elements[0], elements[1], elements[2], logg=logg,atm_type=target_row['diff_atm_type'])
            line= plt.plot(dp_el3el2, dp_el1el2, marker='o', label="teff="+str(teff)+'K,logg='+str(logg), color=color_dict[name], alpha=dp_alpha)

            slope=(dp_el1el2-np.roll(dp_el1el2,1))/(dp_el3el2-np.roll(dp_el3el2,1))
            add_arrow(line[0],position=arr_naca[0], slope=slope[0])
            #print('Slope:', slope)
        elif (plot_type=='arrow'):
            print("successfully plot type arrow happening")
            times=t_step
            #arrow_endy,arrow_endx=acorr.el1el2_DP_el3el2_ftimes(teff, target_row[string1], target_row[string2],times, elements[0], elements[1], elements[2], logg=logg,atm_type=target_row['diff_atm_type'])
            #print('arrow_endy',arrow_endy,'arrow_endx',arrow_endx)
            #plt.plot(arrow_endx,arrow_endy,marker='o')
            arrow_endy,arrow_endx=acorr.el1el2_DP_el3el2_ftimes(teff, target_el1el2, target_el3el2,times, elements[0], elements[1], elements[2], logg=logg,atm_type=target_row['diff_atm_type'])
            print(string1,arrow_endy)
            print(string2,arrow_endx)
            ##ypoints=np.linspace(target_el1el2,arrow_endy,arrow_segs)
            ##xpoints=np.linspace(target_el3el2,arrow_endx,arrow_segs)
            #dx=arrow_endx-target_el3el2
            #dy=arrow_endy-target_el1el2
            ##dx=arrow_endx-xpoints[-2]
            ##dy=arrow_endy-ypoints[-2]
            ##print()
            ##def get_segs(points):
            ##    return np.vstack([points,np.roll(points,1)]).T[1:]
            ##x_segs=get_segs(xpoints)
            ##y_segs=get_segs(ypoints)
            #print('x_segs',x_segs)
            #color_array=np.empty_like(ypoints,dtype=str)
            #color_array[:]=color_dict[name]
            ##alpha_vals=np.linspace(alpha_range[0],alpha_range[1],arrow_segs)
            #print('alpha_vals',alpha_vals)
            #for x,y,alpha in zip(x_segs, y_segs,alpha_vals):
                #plt.plot(x,y,color=color_dict[name],alpha=alpha,linewidth=arrow_line)
                #plt.plot(x,y,color=color_dict[name],alpha=alpha)
            #plt.arrow(xpoints[-2],ypoints[-2],dx,dy,color=color_dict[name],width=arrow_width, alpha=alpha_range[1],length_includes_head=True )
            try:
                #plt.arrow(target_el3el2,target_el1el2,arrow_endx-target_el3el2,arrow_endy-target_el1el2,color=target_row['plot_color'],width=arrow_width, alpha=alpha_range[0],length_includes_head=True,linewidth=0 )
                plt.arrow(target_el3el2,target_el1el2,arrow_endx-target_el3el2,arrow_endy-target_el1el2,color=target_row['plot_color'],width=arrow_width, alpha=alpha_range[0],length_includes_head=True,linewidth=0 )
                print(string2+ ' decreasing phase arrow length:',arrow_endx-target_el3el2)
                print(string1+' decreasing phase arrow length:', arrow_endy-target_el1el2)

            except ValueError:
                print('\nPoint for',label, "can't draw arrow because no data\n")
        else:
            print('\nplot_type not recognized', plot_type,'\n')
    else:
        print('show_dp disabled for object',target_row['show_dp'])
    print('\n\n',target_row['name'])
    #plt.errorbar(target_el3el2,target_el1el2,xerr=el3el2_err,yerr=0.3, uplims=True, marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']],linestyle='None')
    #plt.errorbar(target_el3el2,target_el1el2,label=label,uplims=True, marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']],linestyle='None')
    print("about to try")
    #if target_row[elements[0].lower()+'/'+elements[1].lower()+'_err'] > 0.:
    #    plt.errorbar(target_el3el2,target_el1el2, label=fs.fix_display_string(label), marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']], linestyle='None')
    #    plt.errorbar(target_el3el2,target_el1el2,xerr=el3el2_err,yerr=el1el2_err, marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']], linestyle='None')
    #elif target_row[elements[2].lower()+'/'+elements[1].lower()+'_err'] < 0.001:
    #    plt.errorbar(target_el3el2,target_el1el2,xerr=0.3,yerr=0.3, uplims=True, xuplims=True, marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']],linestyle='None')
    #    plt.errorbar(target_el3el2,target_el1el2,label=label,uplims=True, marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']],linestyle='None')
    
    #else:
    #    plt.errorbar(target_el3el2,target_el1el2,xerr=el3el2_err,yerr=0.3, uplims=True, marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']],linestyle='None')
    #    plt.errorbar(target_el3el2,target_el1el2,label=label,uplims=True, marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']],linestyle='None')
    plot_wd_errorbar(target_el3el2, target_el1el2,  el3el2_err, el1el2_err,row, selected_marker=plot_marker, markersize=markersize, label=label, color=target_row['plot_color'])
    
    print("\n\n***\n\nPlotting concluded for", name,'\n\n***\n')
    return

In [15]:
npoints=100

def plot_objects_el1el2el3(elements=['K','Ca','Na']):
    
    index_val=0
    for row in use_bodies_table:
        object_el1el2=row[elements[0].lower()+'/'+elements[1].lower()]
        object_el3el2=row[elements[2].lower()+'/'+elements[1].lower()]
        #print(row['name'],object_el1el2, object_el3el2)
        #print(type(object_el1el2), object_el1el2, object_el1el2.dtype)
        #print("object_el1el2==False",object_el1el2==False)
        el1el2_clear=False
        el3el2_clear=False
        try:
            object_el1el2.mask
            el1el2_clear=True
        except AttributeError:
            try:
                print(object_el3el2.mask)
                el3el2_clear=True
            except AttributeError:
                #plt.text(object_el3el2, object_el1el2,row['name'],fontsize=figure_text_size)
                
                if show_all_ssobj_names==True:
                    plt.text(object_el3el2, object_el1el2,row['name'],fontsize=figure_text_size)
                elif row['show_name']==1:
                    plt.text(object_el3el2, object_el1el2,row['name'],fontsize=figure_text_size)
                else:
                    pass
                
                if "CI" in row['name']:
                    plt.errorbar(object_el3el2, object_el1el2, color=met_color, marker=met_marker, markersize=ci_size, linestyle='None')
                elif "Cont" in row['name']:
                    plt.errorbar(object_el3el2,object_el1el2,color=met_color,marker=r'$\oplus$',markersize=ci_size+2,linestyle='None')
                elif index_val==0:
                    plt.errorbar(object_el3el2, object_el1el2, color=met_color,marker=met_marker , label='Solar System Body',linestyle='None',markersize=met_size)
                    index_val+=1


                else:
                    #print('else statement so should be plotting',row['name'],object_el1el2, object_el3el2)
                    plt.errorbar(object_el3el2, object_el1el2, color=met_color, marker=met_marker, linestyle='None',markersize=met_size)
        #index_val+=1
    
    return






In [16]:
def generate_el1el2el3_plot(elements=["K","Ca","Na"],leg_loc='best', show_legend=True,fig_size='default'): 
    spt.initiate_science_plot()

    plot_type='arrow'
    #plot_type='nonsense'
    #plt.figure(figsize=(7.25,7.25),constrained_layout=True)
    if fig_size=='default':
        #plt.figure(figsize=(4.75,4.75),constrained_layout=True)
        plt.figure(figsize=(7.25,7.25),constrained_layout=True)
    else:
        plt.figure(figsize=fig_size,constrained_layout=True)
    if elements== ['Li','Ca',"Na"]:
        #spt.start_ApJ_fig(width_cols=1,width_height=[1.5+2.75, 0.3+4.4])
        spt.start_ApJ_fig(width_cols=2,width_height=[1.5+3., 0.3+4.5])
    else:
        spt.start_ApJ_fig(width_cols=2)





    t_max=10
    count=0
    plot_objects_el1el2el3(elements=elements)


    for j1644_row in wd_abund_table:

        print('\n\n',j1644_row['name'])
        #if count==2:
        #    break
        #t_step=step_dict[j1644_row['name']]
        plot_wd_el1el2el3(j1644_row, logg='default', t_step=t_step, naca_min=-4.0, t_max=t_max, t_step_units='Ca', plot_type=plot_type, elements=elements)
        count+=1
        print("\n\n=====\n",count,"\n====\n\n")
    #plt.errorbar(j1644_row['na/ca'],j1644_row['k/ca'],xerr=j1644_row['na/ca_err'],yerr=0.3, uplims=True,label=j1644_row['name'], marker=wd_marker, color=color_dict[j1644_row['name']], markersize=wd_size)
    print('\n\n***\nWhite dwarf plotting fully concluded\n****\n\n')

    if elements==['Li','Ca',"Na"]:
        #plt.xlim(-2.75,1.25)
        #plt.ylim(-4.5,0.0)
        #plt.axhline(y=-2.3, linestyle=':',color='k')
        #plt.text(-2.5,-2.25,'GALEXJ2339 photospheric Be-implied spalled Li level from CI')
        #plt.axhline(y=-1.5, linestyle=':',color='k') #pages 30-32 of General Clemens XI for calculation
        #plt.text(-2.5,-1.45,'GALEXJ2339 photospheric Be-implied spalled Li level from CI')
        #J2339_LiCa_array=get_spalled_LiCa(n_sigma=1.)
        #J2339_LiCa_array=get_spalled_LiCa(n_sigma=3.)
        J2339_LiCa_array=get_spalled_LiCa(n_sigma=spall_n_sigma)
        plt.axhspan(J2339_LiCa_array[1],J2339_LiCa_array[2],alpha=0.1, color='k')
        plt.axhline(y=J2339_LiCa_array[0], linestyle='--',color='k')
        #plt.text(-2.7,J2339_LiCa_array[0]+0.05,'GALEXJ2339 photospheric Be-implied spalled Li level from CI')
        #plt.text(-2.7,J2339_LiCa_array[0]+0.05,'GALEXJ2339 photospheric Be-implied spalled Li level from CI ('+str(int(spall_n_sigma))+'-$\sigma$)')
        #plt.xlim(-2.75,1.5)
        plt.xlim(-3.,1.5)
        #plt.ylim(-4.4,0.3)
        plt.ylim(-4.5,0.3)
    elif elements==['K','Ca',"Na"]:
        plt.xlim(-2.75,1.5)
        plt.ylim(-4.25,0.75)
    elif elements==['Li','Na',"K"]:
        plt.xlim(-1.8,0.05)
        plt.ylim(-3.75,-1.0)
    else:
        pass

    #plt.xlabel('log(Na/Ca)')
    #plt.ylabel('log(K/Ca)')
    plt.xlabel('log('+elements[2]+'/'+elements[1]+')')
    plt.ylabel('log('+elements[0]+'/'+elements[1]+')')
    if show_legend:
        #plt.legend(loc=leg_loc, fontsize=7)
        plt.legend(loc=leg_loc)
    else:
        pass
    #plt.xlim(-0.5,0.5)
    #plt.ylim(-1.75,-0.75)
    #plt.xlim(4.5,0)
    #plt.ylim(4.0,1.0)
    

    if savefig:
        print(os.getcwd())
        os.chdir(figure_output_dir)
        print(os.getcwd())
        start = time.time()
        print(start)
        time_string=str(start).split('.')[0]
        plt.savefig(elements[0]+elements[1]+'_vs_'+elements[2]+elements[1]+'_'+time_string+'.pdf')#plt.grid(True)
        print("Figure saved")
    else:
        pass

    plt.show()
    return






In [17]:
#generate_el1el2el3_plot(elements=["Na","Li","Ca"])

In [45]:

generate_el1el2el3_plot(elements=["K","Ca","Na"], leg_loc='upper left', show_legend=True)





 WDJ1644-0449
starting plotting effort for WDJ1644-0449
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3853.3674470482597 7.7394664041915355
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3853.3674470482597 7.7394664041915355
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3853.3674470482597 7.7394664041915355
tau_K-tau_Ca -0.004214238268957577 +/- 0.20131700224054352
tau_Na-tau_Ca

using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
4577.3123313786145 8.769216370277743
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
4577.3123313786145 8.769216370277743
tau_K-tau_Ca 0.0022387039664518655 +/- 0.2002503645918981
tau_Na-tau_Ca 0.2508063265255101 +/- 0.20097046921501094
target_ssp K Ca 0.9195633780757807
target_ssp Na Ca 0.5182552598516321
dist ssp K Ca 0.9191270289666291 0.9177612960335482 0.2002503645918981
dist ssp Na Ca 0.5170773495304614 0.5201741149814816 0.2643080396647404
SSP log(K/Ca) = 0.9195633780757807 +/- -100.0
SSP log(Na/Ca) = 0.5182552598516321 +/- 0.2643080396647404
t_step_units is not Myr, meaning it's some diffusion timescale multiple
so t_step=  5 * tau_ Ca
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
New t_step: 0.5429

Figure saved


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:93: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


In [44]:

generate_el1el2el3_plot(elements=["Li","Ca","Na"],leg_loc='upper left')



 WDJ1644-0449
starting plotting effort for WDJ1644-0449
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3339.1407382806556 8.021629769473037
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3339.1407382806556 8.021629769473037
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3339.1407382806556 8.021629769473037
tau_Li-tau_Ca 0.5537724632767532 +/- 0.2037411684266692
tau_Na-tau_Ca 0.238

using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
4506.985524790889 8.770346052345522
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
4506.985524790889 8.770346052345522
tau_Li-tau_Ca 0.6409780645194632 +/- 0.19990997564050822
tau_Na-tau_Ca 0.2492027421566951 +/- 0.20121518110200784
target_ssp Li Ca -0.8230575771118505
target_ssp Na Ca 0.5182552598516321
dist ssp Li Ca -0.8237003400722763 -0.823398674090569 0.2718645561948783
dist ssp Na Ca 0.5208829677524197 0.5210210185411015 0.26311724201526443
SSP log(Li/Ca) = -0.8230575771118505 +/- 0.2718645561948783
SSP log(Na/Ca) = 0.5182552598516321 +/- 0.26311724201526443
t_step_units is not Myr, meaning it's some diffusion timescale multiple
so t_step=  5 * tau_ Ca
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
N

Figure saved


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:93: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


In [20]:
#generate_el1el2el3_plot(elements=["Li","Na","K"],fig_size=(5.5,5.5))

In [21]:
#generate_el1el2el3_plot(elements=["Li","Na","Ca"])

In [22]:
#generate_el1el2el3_plot(elements=["Na","Mg","Ca"])

In [23]:
#generate_el1el2el3_plot(elements=["Li","Ca","K"])

In [24]:
#generate_el1el2el3_plot(elements=["Li","K","Na"])

In [25]:
#generate_el1el2el3_plot(elements=["Ca","Fe","Mg"],fig_size=(5.5,5.5))

In [26]:
generate_el1el2el3_plot(elements=["Mg","Ca","Na"],leg_loc='best')



 WDJ1644-0449
starting plotting effort for WDJ1644-0449
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3665.0150300852747 7.666602103536015


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:83: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order)
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:136: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order, subok=True)
/Users/BenKaiser/Desktop/radial_velocity_calculations/abundance_corrections.py:295: UserWarning: Warning: converting a masked element to nan.
  el1el2_dist=np.random.normal(loc=target_el1el2,scale=wd_row[string1+'_err'],size=n_points)


using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3665.0150300852747 7.666602103536015
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3665.0150300852747 7.666602103536015
tau_Mg-tau_Ca 0.24654884482574793 +/- 0.19917378362103413
tau_Na-tau_Ca 0.24189580020335738 +/- 0.19756023941448478
target_ssp Mg Ca --
target_ssp Na Ca -0.1611582243037067
dist ssp Mg Ca nan nan nan
dist ssp Na Ca -0.16265295437234897 -0.16039575741372614 0.26362524093654416
SSP log(Mg/Ca) = -- +/- nan
SSP log(Na/Ca) = -0.1611582243037067 +/- 0.26362524093654416
t_step_units is not Myr, meaning it's some diffusion timescale multiple
so t_step=  5 * tau_ Ca
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
New t_step: 177.1357601921038 Myr
New t_max: 354.2715203842076 Myr
successfully plot t

using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
5028.244238400539 7.932316676627664
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
5028.244238400539 7.932316676627664
tau_Mg-tau_Ca 0.2275810463515534 +/- 0.20058466890095492
tau_Na-tau_Ca 0.21868633770267665 +/- 0.20067565929099082
target_ssp Mg Ca --
target_ssp Na Ca 0.4390982288533606
dist ssp Mg Ca nan nan nan
dist ssp Na Ca 0.4429150030714196 0.4406398732684869 0.23723277432711048
SSP log(Mg/Ca) = -- +/- nan
SSP log(Na/Ca) = 0.4390982288533606 +/- 0.23723277432711048
t_step_units is not Myr, meaning it's some diffusion timescale multiple
so t_step=  5 * tau_ Ca
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
New t_step: 25.67263591714957 Myr
New t_max: 51.34527183429914 Myr
successfully plot type arrow

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:93: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


In [27]:
generate_el1el2el3_plot(elements=["Cr","Ca","Na"],leg_loc='best')



 WDJ1644-0449
starting plotting effort for WDJ1644-0449
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3681.7989544747034 7.929982729145764


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:83: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order)
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:136: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order, subok=True)
/Users/BenKaiser/Desktop/radial_velocity_calculations/abundance_corrections.py:295: UserWarning: Warning: converting a masked element to nan.
  el1el2_dist=np.random.normal(loc=target_el1el2,scale=wd_row[string1+'_err'],size=n_points)


using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3681.7989544747034 7.929982729145764
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3681.7989544747034 7.929982729145764
tau_Cr-tau_Ca -0.17743549435801517 +/- 0.19797474347530367
tau_Na-tau_Ca 0.24287075268193498 +/- 0.20297448569453774
target_ssp Cr Ca --
target_ssp Na Ca -0.1611582243037067
dist ssp Cr Ca nan nan nan
dist ssp Na Ca -0.16654931419566255 -0.16333074938379233 0.26173494323052315
SSP log(Cr/Ca) = -- +/- nan
SSP log(Na/Ca) = -0.1611582243037067 +/- 0.26173494323052315
t_step_units is not Myr, meaning it's some diffusion timescale multiple
so t_step=  5 * tau_ Ca
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
New t_step: 177.1357601921038 Myr
New t_max: 354.2715203842076 Myr
successfully plot 

using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
5227.837800296453 8.07162414737619
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
5227.837800296453 8.07162414737619
tau_Cr-tau_Ca -0.15674070603188714 +/- 0.20107384817224902
tau_Na-tau_Ca 0.21878500052603359 +/- 0.19992845573221757
target_ssp Cr Ca 0.01706470767243984
target_ssp Na Ca 0.4390982288533606
dist ssp Cr Ca 0.01608211060225144 0.017445938409661892 0.2443524326780767
dist ssp Na Ca 0.44216390730679783 0.44075130529184314 0.2375797276346584
SSP log(Cr/Ca) = 0.01706470767243984 +/- 0.2443524326780767
SSP log(Na/Ca) = 0.4390982288533606 +/- 0.2375797276346584
t_step_units is not Myr, meaning it's some diffusion timescale multiple
so t_step=  5 * tau_ Ca
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:93: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


In [28]:
generate_el1el2el3_plot(elements=["Fe","Ca","Na"],leg_loc='best')



 WDJ1644-0449
starting plotting effort for WDJ1644-0449
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
4036.419044110941 8.016118202387439


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:83: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order)
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:136: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order, subok=True)
/Users/BenKaiser/Desktop/radial_velocity_calculations/abundance_corrections.py:295: UserWarning: Warning: converting a masked element to nan.
  el1el2_dist=np.random.normal(loc=target_el1el2,scale=wd_row[string1+'_err'],size=n_points)


using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
4036.419044110941 8.016118202387439
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
4036.419044110941 8.016118202387439
tau_Fe-tau_Ca -0.20653945688317882 +/- 0.2011623085930538
tau_Na-tau_Ca 0.24226733337328268 +/- 0.199497356911952
target_ssp Fe Ca --
target_ssp Na Ca -0.1611582243037067
dist ssp Fe Ca nan nan nan
dist ssp Na Ca -0.16448142157707119 -0.16228590539924997 0.26362583151001917
SSP log(Fe/Ca) = -- +/- nan
SSP log(Na/Ca) = -0.1611582243037067 +/- 0.26362583151001917
t_step_units is not Myr, meaning it's some diffusion timescale multiple
so t_step=  5 * tau_ Ca
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
New t_step: 177.1357601921038 Myr
New t_max: 354.2715203842076 Myr
successfully plot type 

using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
5087.2927620650125 8.306996868959267
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
5087.2927620650125 8.306996868959267
tau_Fe-tau_Ca -0.1813034569237586 +/- 0.19989007742720571
tau_Na-tau_Ca 0.2234841578269593 +/- 0.1992483320821016
target_ssp Fe Ca 1.2245564046184763
target_ssp Na Ca 0.4390982288533606
dist ssp Fe Ca 1.2259347157979563 1.2222733111296842 0.30371541538119023
dist ssp Na Ca 0.4356977105369757 0.43722718313092307 0.2365152243303988
SSP log(Fe/Ca) = 1.2245564046184763 +/- 0.30371541538119023
SSP log(Na/Ca) = 0.4390982288533606 +/- 0.2365152243303988
t_step_units is not Myr, meaning it's some diffusion timescale multiple
so t_step=  5 * tau_ Ca
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
N

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:93: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


In [29]:
#generate_el1el2el3_plot(elements=["Fe","Ca","Mg"])


In [30]:
#generate_el1el2el3_plot(elements=["Cr","Ca","K"])

In [31]:
#generate_el1el2el3_plot(elements=["Fe","Ca","Na"])

In [32]:
#generate_el1el2el3_plot(elements=["Fe","Ca","Cr"])

In [33]:
#generate_el1el2el3_plot(elements=["Li","Ca","Mg"])

In [34]:
wd_abund_table.columns

<TableColumns names=('name','display_name','modeler','cooling_model','teff','teff_err','logg','logg_err','m_wd','h/he','h/he_err','li/he','li/he_err','be/he','be/he_err','na/he','na/he_err','mg/he','mg/he_err','k/he','k/he_err','ca/he','ca/he_err','cr/he','cr/he_err','fe/he','fe/he_err','li/ca','li/ca_err','na/ca','na/ca_err','mg/ca','mg/ca_err','k/ca','k/ca_err','cr/ca','cr/ca_err','fe/ca','fe/ca_err','li/na','li/na_err','k/na','k/na_err','ca/na','ca/na_err','li/k','li/k_err','na/k','na/k_err','ca/fe','ca/fe_err','mg/fe','mg/fe_err','na/mg','na/mg_err','ca/mg','ca/mg_err','ca/cr','ca/cr_err','k/cr','k/cr_err','cr/fe','cr/fe_err','na/li','na/li_err','ca/li','ca/li_err','atm_type','diff_atm_type','log_q','age','age_minus','age_plus','med_age','med_age_minus','med_age_plus','vtan_lsr','vtan_lsr_err_lo','vtan_lsr_err_hi','v','uw2','v_err_lo','v_err_hi','uw2_err_lo','uw2_err_hi','plot_color','age_dist_file','show','show_li_evo','show_geo','show_dp','thin_disk','thick_disk','halo','show_age

In [35]:
np.log10(3.621980492301957*1e6/5)

5.85997610257288

In [36]:
np.log10(17.4506343576774*1e6/5)

6.542841214525449

In [37]:
6.26-5.86

0.39999999999999947

In [38]:
6.56-6.32

0.23999999999999932

In [39]:
0.3999-0.23999

0.15990999999999997

In [40]:
np.log10(9.79e3)-np.log10(2.91e3)

0.5268897028172304

In [41]:
np.log10(4.52e3)-np.log10(2.91e3)

0.19124544582547465

In [42]:
np.log10(2.93e3)-np.log10(2.91e3)

0.0029746313682021963

In [43]:
np.e**5

148.41315910257657